# 02 – Líkön

Í þessari minnisbók þjálfum við nokkur vélnámslíkön og berum þau saman á sömu prófunargögnunum:

1. **Allir flokkar með sömu líkur** (*uniform baseline*) — gefur 1/7 líkur á hvern flokk. Þetta er ein einfaldasta nálgunin, hreint gisk. Við höfum þessa nálgun með til að sjá hversu vel önnur líkön standa sig í samanburði við það að giska.
2. **Tíðasta gildið** (*modal baseline*) — giskar alltaf á stærsta flokk. Þessi nálgun stendur sig betur en hreint gisk en við viljum að sjálfsögðu gera betur en þessi nálgun og því gott að hafa hana með til viðmiðunar.
3. **Ákvarðanatökutré** (*decision tree*) — grunnurinn fyrir önnur tréreiknirit; einfalt og skiljanlegt.
4. **Random forest** — samsettur skógur ákvarðanatökutrjáa; klassískt vélnám. Erfiðara að skilja hvað er að gerast.
5. **Gradient boosting** (`HistGradientBoostingClassifier`) — nútímalegra samsett líkan.
6. **AutoGluon** — sjálfvirkt vélnám (*AutoML*) sem þjálfar mörg líkön og sameinar þau með sjálfvirkum hætti. Við gefum aðferðinni 15 mínútur til að finna besta líkanið en ef þið gefið því meiri tíma eru ágætar líkur á því að niðurstaðan verði betri.

**Tvær keyrslur** fyrir random forest, gradient boosting og AutoGluon:

- **Grunnbreytur** (*base features*) — félagsleg staða + málefnaviðhorf + frammistaða stjórnvalda.
- **Útvíkkað** (*enriched features*) — bætir málefnaeign (*issue ownership*) við. Þessar breytur eru mjög nálægt flokksvali og við skoðum hvaða áhrif það hefur á niðurstöðuna.

**Úttak:** Eftir að hafa keyrt þetta skjal eru líkönin vistuð í `.pkl` skrár undir `models/` og myndir og önnur gögn eru vistuð í möppunum `figures/confusion_matrix_*.png`, `figures/accuracy_table.csv`, `figures/chapter_numbers.csv`, og `data/processed/predictions.parquet`. Við notum líkönin í þriðja skjalinu til að túlka það hvernig líkönin eru að virka.

In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    top_k_accuracy_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = PROJECT_ROOT / "data" / "processed"
DATA_PATH = PROJECT_ROOT / "Kjósendakönnun" / "ÍSKOS kjósenda 2024" / "ISKOS2024_med_aukabreytum_20260318.sav"
MODELS = PROJECT_ROOT / "models"
FIGURES = PROJECT_ROOT / "figures"
MODELS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT}")

# EDIT-2026-04-13 imports
# Make src/iskos_labels.py importable from this notebook
import sys
_SRC = str((PROJECT_ROOT / "src").resolve())
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)
from iskos_labels import IskosLabels  # noqa: E402
labels = IskosLabels.from_sav(DATA_PATH, feature_groups_path=PROC / "feature_groups.json")
print(f"Loaded {len(labels.variable_labels)} variable labels.")

## 1 — Hlöðum inn gögnum og lýsigögnum

Við hlöðum inn `train.parquet`, `test.parquet`, `feature_groups.json` og `target_labels.json` sem voru útbúin í `01_data_prep.ipynb`.

In [ ]:
train_df = pd.read_parquet(PROC / "train.parquet")
test_df = pd.read_parquet(PROC / "test.parquet")

with (PROC / "feature_groups.json").open() as f:
    fg = json.load(f)
with (PROC / "target_labels.json").open() as f:
    tl = json.load(f)

BASE_FEATURES = fg["base_features"]
ENRICHED_FEATURES = fg["enriched_features"]
NOMINAL_COLS = set(fg["nominal_columns"])
NUMERIC_COLS = set(fg["numeric_columns"])

# Class order we want everywhere (largest parties first, merged left last)
CLASS_ORDER = ["S", "D", "C", "F", "M", "B", "Vinstri_sameinad"]
CLASS_LABELS_IS = [tl["target_class_labels_is"][c] for c in CLASS_ORDER]

y_train = train_df["y"]
y_test = test_df["y"]

print(f"Train: {train_df.shape[0]:,} rows, Test: {test_df.shape[0]:,} rows")
print(f"Base features: {len(BASE_FEATURES)}")
print(f"Enriched features: {len(ENRICHED_FEATURES)}")
print(f"Class order: {CLASS_ORDER}")

## 2 — Forvinnsla gagna

Við notum `ColumnTransformer` sem gerir tvennt:

- **Tölulegar breytur** — ef gildi vantar setjum við miðgildið í staðinn (miðgildi reiknað eingöngu á þjálfunargögnum; þetta kemur í veg fyrir *gagnaleka*).
- **Nafnbreytur** — one-hot kóðun með `handle_unknown="ignore"` svo ný gildi í prófunarhluta brjóta ekki kóðunina. Þar sem við breyttum `NaN` í flokkinn `"Missing"` í `01_data_prep.ipynb` þarf ekki að höndla gildi sem vantar sérstaklega.

Við framkvæmum **forvinnsluna tvisvar**, einu sinni fyrir grunnbreytur og svo aftur fyrir útvíkkaðar breytur. Það auðveldar okkur að skipta gögnunum út síðar þegar við þjálfum líkönin.

In [ ]:
def build_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    """Return a ColumnTransformer for a given feature set.

    Numeric columns get median imputation; nominal (categorical) columns get
    one-hot encoding. The preprocessor is fit by each pipeline/GridSearchCV
    on training folds only.
    """
    numeric = [c for c in feature_cols if c in NUMERIC_COLS]
    nominal = [c for c in feature_cols if c in NOMINAL_COLS]
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                nominal,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

# Pre-build and fit shared preprocessors (used for non-sklearn use cases like SHAP)
preprocessor_base = build_preprocessor(BASE_FEATURES).fit(train_df[BASE_FEATURES])
preprocessor_enriched = build_preprocessor(ENRICHED_FEATURES).fit(train_df[ENRICHED_FEATURES])

joblib.dump(preprocessor_base, MODELS / "preprocessor_base.pkl")
joblib.dump(preprocessor_enriched, MODELS / "preprocessor_enriched.pkl")

# Shape of transformed matrices, for sanity
Xb = preprocessor_base.transform(train_df[BASE_FEATURES])
Xe = preprocessor_enriched.transform(train_df[ENRICHED_FEATURES])
print(f"X_base       transformed: {Xb.shape}")
print(f"X_enriched   transformed: {Xe.shape}")
print(f"Base feature names (first 10): {list(preprocessor_base.get_feature_names_out())[:10]}")

# EDIT-2026-04-13 missing-counts
### Eyður í forspárbreytum

Áður en við notum leiðsluna er gott að sjá **hve margar eyður eru í
hverri breytu**. Þetta gefur okkur tilfinningu fyrir gæðum gagnanna og
hvar miðgildisímiðlunin (eða `Missing`-flokkurinn) mun hafa mest áhrif.

In [ ]:
# EDIT-2026-04-13 missing-counts
# Per-column missing counts on the training set, sorted by amount.
# `Categorical` columns can't be NaN here (they were converted to a
# 'Missing' level in notebook 01) — for those we count the 'Missing'
# category instead.
feature_cols = ENRICHED_FEATURES
rows = []
for col in feature_cols:
    s = train_df[col]
    if pd.api.types.is_categorical_dtype(s):
        missing_n = int((s == "Missing").sum())
        kind = "categorical"
    else:
        missing_n = int(s.isna().sum())
        kind = "numeric"
    rows.append({
        "Breyta": col,
        "Lýsing": labels.short_question(col),
        "Tegund": kind,
        "Vantar": missing_n,
        "Hlutfall (%)": round(100 * missing_n / len(s), 1),
    })
missing_df = pd.DataFrame(rows).sort_values("Vantar", ascending=False).reset_index(drop=True)
print(f"Þjálfunarúrtak: {len(train_df):,} svarendur, {len(feature_cols)} forspárbreytur.")
print(f"Breytur með a.m.k. eitt vantandi gildi: "
      f"{(missing_df['Vantar'] > 0).sum()}/{len(missing_df)}")
missing_df

## 3 — Útbúum leið til að safna saman niðurstöðum

Við söfnum niðurstöðum allra líkana í eina töflu sem við skrifum út í lokin. Í henni eru eftirfarandi dálkar:

- **Heiti líkans** og **breytusafn** (grunn/útvíkkað).
- **Meðhöndlun** (`naive` eða `balanced`) — seinna gerum við *afnámstilraun* (e. *ablation study*) þar sem við skoðum áhrif þess að taka tillit til ójafnvægis í gögnunum. Þetta þarf að gera því sumir flokkar eru mun stærri en aðrir sem getur myndað bjaga í líkönunum svo þau giski of oft á stærri flokkana.
- **Nákvæmni**, **Topp-2** og **Topp-3 nákvæmni**.
- **Jöfnuð nákvæmni** (e. *balanced accuracy*) — meðaltal nákvæmni mælingarinnar yfir flokkana sjö. Þetta er góður mælikvarði fyrir gögn sem eru ekki jafndreifð eins og í okkar tilviki.
- **Macro-F1** — meðaltal F1 mælingar yfir flokkana sjö.
- **Per-class recall** fyrir alla sjö flokkana, vistað í sérskrá (`per_class_recall.csv`) til samanburðar. Recall hefur verið þýtt sem endurheimt á íslensku og mælir hversu vel okkur tekst að finna einstaklinga sem kjósa tiltekinn flokk. Ef endurheimt fyrir flokk fólksins er 35% þýðir það að líkaninu tekst að flokka rétt 35% þeirra sem kusu flokk fólksins en fyrir restina giskaði líkanið á eitthvað annað.

In [ ]:
results: list[dict] = []
per_class_recall_rows: list[dict] = []

# Class order used consistently when reporting per-class recall
RECALL_CLASS_ORDER = ["S", "D", "C", "F", "M", "B", "Vinstri_sameinad"]


def record(
    name: str,
    feature_set: str,
    y_true,
    y_pred,
    y_proba=None,
    classes=None,
    treatment: str = "naive",
):
    """Append one row (per model) to `results` and 7 rows to `per_class_recall_rows`.

    `classes` must correspond to the column order of `y_proba` (it can be
    unsorted; we sort internally for `top_k_accuracy_score`).
    `treatment` is "naive" (no imbalance correction) or "balanced"
    (class_weight / sample_weight applied during training).
    """
    # Overall metrics
    row = {
        "model": name,
        "features": feature_set,
        "treatment": treatment,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_proba is not None:
        # top_k_accuracy_score requires `labels` to be sorted
        classes_sorted = sorted(classes)
        if list(classes) != classes_sorted:
            col_order = [classes.index(c) for c in classes_sorted]
            y_proba = y_proba[:, col_order]
        row["top2_accuracy"] = top_k_accuracy_score(y_true, y_proba, k=2, labels=classes_sorted)
        row["top3_accuracy"] = top_k_accuracy_score(y_true, y_proba, k=3, labels=classes_sorted)
    results.append(row)

    # Per-class recall (one row per class for long-format CSV)
    recalls = recall_score(
        y_true, y_pred, labels=RECALL_CLASS_ORDER, average=None, zero_division=0,
    )
    for cls, r in zip(RECALL_CLASS_ORDER, recalls):
        per_class_recall_rows.append({
            "model": name,
            "features": feature_set,
            "treatment": treatment,
            "class": cls,
            "recall": float(r),
        })
    return row


# Storage for per-model test predictions (written to data/processed/predictions.parquet)
predictions: dict[str, np.ndarray] = {}

# Sample weights used by HistGB-balanced and AutoGluon-balanced in section 17-18.
# Formula: n_samples / (n_classes * bincount(y)), identical to class_weight="balanced".
sample_weight_balanced = compute_sample_weight("balanced", y_train)
print(f"Balanced sample weights — min: {sample_weight_balanced.min():.3f}, "
      f"max: {sample_weight_balanced.max():.3f}")

## 4 — Jafnar líkur (hreint gisk)

Einfaldasta hugsanlega nálgunin: gefur öllum sjö flokkum sömu líkur (1/7 ≈ 14,3%) og spáir handahófskennt út frá þeim líkindum. Þetta er viðmiðið sem öll önnur líkön verða að slá út.

In [ ]:
# A true uniform prediction scheme would pick a random class per row.
# We evaluate the expected accuracy analytically (1/7) plus an empirical draw
# for the confusion matrix below.
rng = np.random.default_rng(RANDOM_STATE)
# Use sorted class order here so it aligns with top_k_accuracy_score's expectations
CLASS_ORDER_SORTED = sorted(CLASS_ORDER)
uniform_pred = rng.choice(CLASS_ORDER_SORTED, size=len(y_test))
uniform_proba = np.full((len(y_test), len(CLASS_ORDER_SORTED)), 1 / len(CLASS_ORDER_SORTED))

record("Uniform baseline", "–", y_test, uniform_pred, uniform_proba, CLASS_ORDER_SORTED)
predictions["uniform"] = uniform_pred
print(f"Uniform baseline empirical accuracy: {accuracy_score(y_test, uniform_pred):.3f}")
print(f"Analytical expectation              : {1/7:.3f}")

## 5 — Tíðasta gildið

Líkanið giskar alltaf á þann flokk sem var algengastur í þjálfunargögnum (Samfylkinguna, sem er stærst í ÍSKOS úrtakinu). Þetta er næsta einfaldasta viðmiðið.

In [ ]:
modal = DummyClassifier(strategy="most_frequent")
modal.fit(train_df[BASE_FEATURES], y_train)
modal_pred = modal.predict(test_df[BASE_FEATURES])
modal_proba = modal.predict_proba(test_df[BASE_FEATURES])

# DummyClassifier.classes_ is already sorted
record("Modal baseline (always top class)", "–", y_test, modal_pred, modal_proba, list(modal.classes_))
predictions["modal"] = modal_pred
print(f"Modal baseline accuracy: {accuracy_score(y_test, modal_pred):.3f}")
print(f"Always predicts:         {modal_pred[0]} ({tl['target_class_labels_is'][modal_pred[0]]})")

## 6 — Ákvarðanatökutré

Ákvarðanatré **klýfur** gögnin endurtekið í undirhópa. Við hvert skipti sem gögnin eru klofin velur það breytu og viðmiðunargildi sem gera hópana sem **einsleitasta** — þ.e. sem flestir kjósendur í sama hópi kjósa sama flokk.

Við þjálfum eitt lítið tré (`max_depth=3`, því það er auðveldast að teikna það upp og skilja það) og berum svo nákvæmni þess saman við djúpt tré (`max_depth=None`) til að sýna *ofmátun* (*overfitting*), þ.e., stærri tré þurfa ekkert endilega að vera betri en lítil tré.

Neðstu hnútarnir í trénu eru kallaðir lauf og þar eru ákvarðanir teknar.

In [ ]:
tree_pipe = Pipeline([
    ("prep", build_preprocessor(BASE_FEATURES)),
    ("clf", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)),
])
tree_pipe.fit(train_df[BASE_FEATURES], y_train)
tree_pred = tree_pipe.predict(test_df[BASE_FEATURES])
tree_proba = tree_pipe.predict_proba(test_df[BASE_FEATURES])

record("Decision tree (max_depth=3)", "base", y_test, tree_pred, tree_proba, list(tree_pipe.classes_))
predictions["decision_tree"] = tree_pred
joblib.dump(tree_pipe, MODELS / "decision_tree.pkl")
print(f"Decision tree (max_depth=3) accuracy: {accuracy_score(y_test, tree_pred):.3f}")

In [ ]:
# EDIT-2026-04-13 tree-readable-names
# Visualize the tree — saved to figures/decision_tree.png with readable Icelandic names.
# The preprocessor's get_feature_names_out() returns codes like "educationflo_80";
# we pass each through labels.readable() to get e.g. "Menntun: MA, MSc eða sambærilegt".
feature_names_base = tree_pipe.named_steps["prep"].get_feature_names_out()
feature_names_readable = [labels.readable(name) for name in feature_names_base]

# Map class codes (S, D, ...) to Icelandic party names for the leaf labels
class_names_readable = [
    tl["target_class_labels_is"].get(c, c) for c in tree_pipe.classes_
]

fig, ax = plt.subplots(figsize=(22, 12))
plot_tree(
    tree_pipe.named_steps["clf"],
    feature_names=feature_names_readable,
    class_names=class_names_readable,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
)
ax.set_title("Ákvarðanatökutré (max_depth=3) – grunnbreytur")
plt.tight_layout()
fig.savefig(FIGURES / "decision_tree.png", dpi=150, bbox_inches="tight")
plt.show()

### Ofmátun: grunnt tré vs. djúpt tré

Til að sýna hvað gerist þegar tré er leyft að vaxa stjórnlaust berum við saman nákvæmni á þjálfunar- og prófunargögnum fyrir tré með `max_depth=3` og
tré án takmarkana (`max_depth=None`). Djúpa tréð man öll þjálfunargögnin en **alhæfir verr** um þau.

In [ ]:
# EDIT-2026-04-13 depth-sweep
# Sweep tree depth from 3..8 plus an unbounded tree (max_depth=None).
# We track BOTH train and test accuracy so the train-test gap (the
# tell-tale sign of overfitting) is immediately visible.
depth_grid = list(range(3, 9)) + [None]
rows = []
for d in depth_grid:
    pipe = Pipeline([
        ("prep", build_preprocessor(BASE_FEATURES)),
        ("clf", DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)),
    ]).fit(train_df[BASE_FEATURES], y_train)
    rows.append({
        "max_depth": "None" if d is None else str(d),
        "depth_for_x": d if d is not None else max(depth_grid[:-1]) + 2,
        "train_acc": accuracy_score(y_train, pipe.predict(train_df[BASE_FEATURES])),
        "test_acc": accuracy_score(y_test, pipe.predict(test_df[BASE_FEATURES])),
    })
depth_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depth_df["depth_for_x"], depth_df["train_acc"] * 100,
        marker="o", label="Þjálfun", color="#4a90a4", linewidth=2)
ax.plot(depth_df["depth_for_x"], depth_df["test_acc"] * 100,
        marker="o", label="Prófun", color="#c75b5b", linewidth=2)
# x-axis: real depths 3..8 then a labelled "None" tick at the end
ax.set_xticks(depth_df["depth_for_x"])
ax.set_xticklabels(depth_df["max_depth"])
ax.set_xlabel("max_depth")
ax.set_ylabel("Nákvæmni (%)")
ax.set_title("Ofmátun: nákvæmni eykst á þjálfun en flatnar á prófun eftir því sem tréð dýpkar")
ax.legend()
ax.grid(True, alpha=0.3)
# Annotate the train-test gap at the deepest tree
last = depth_df.iloc[-1]
ax.annotate(
    f"Bil: {(last['train_acc'] - last['test_acc']) * 100:.1f} pp",
    xy=(last["depth_for_x"], (last["train_acc"] + last["test_acc"]) * 50),
    xytext=(last["depth_for_x"] - 1.5, (last["train_acc"] + last["test_acc"]) * 50),
    fontsize=10, color="#444",
    arrowprops=dict(arrowstyle="->", color="#444", lw=0.8),
)
plt.tight_layout()
fig.savefig(FIGURES / "tree_depth_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nNákvæmni eftir max_depth (%):")
print(depth_df[["max_depth", "train_acc", "test_acc"]].assign(
    train_acc=lambda d: (d["train_acc"] * 100).round(1),
    test_acc=lambda d: (d["test_acc"] * 100).round(1),
).to_string(index=False))

## 7 — Random forest (grunn + útvíkkað)

*Random forest* er safn margra ákvarðanatökutrjáa. Hvert tré er þjálfað á **slembiúrtaki** af gögnum (e. *bootstrap sample*) og í hvert skipti sem tré klýfur gögnin byggt á breytu kemur einungis **hluta af breytunum** til greina hverju sinni (þetta veldur því að trén verða ekki öll eins). Lokaforspáin er meðaltal eða meirihlutakosning allra trjánna.

Þetta stuðlar að **fjölbreytni** milli trjánna og dregur úr ofmátun. Random forest er gjarnan góð fyrsta tilraun þegar bera þarf saman mörg líkön.

Við þjálfum líkanið tvisvar (grunn og útvíkkað) og notum einfalda **krossgildingarleitargrind** (*cross-validated grid search*) til að velja bestu gildin fyrir `max_depth` og `min_samples_leaf`. Þessar breytur eru kallaðar ofurstikar (e. *hyperparameters*) en það eru stikar sem eru ekki lærðir sjálfkrafa út frá gögnunum. Velja þarf stika sem passa að líkanið geti lært yfir höfuð en ofmáti ekki. Eins og við sáum áður þá skiptir dýpt líkansins máli, ef hún er of mikil þá hefur það neikvæð áhrif á frammistöðu líkansins. Athugið að þessi gildi geta verið mjög háð magni af gögnum, því meira af gögnum sem eru í boði, því dýpri geta trén verið og líkönin því flóknari.

In [ ]:
def fit_rf(feature_cols: list[str], tag: str):
    """Train a RandomForest via GridSearchCV (inside a Pipeline for safe preprocessing)."""
    pipe = Pipeline([
        ("prep", build_preprocessor(feature_cols)),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])
    grid = {
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_leaf": [1, 5, 10],
    }
    search = GridSearchCV(pipe, grid, cv=5, scoring="accuracy", n_jobs=-1)
    search.fit(train_df[feature_cols], y_train)
    best = search.best_estimator_

    y_pred = best.predict(test_df[feature_cols])
    y_proba = best.predict_proba(test_df[feature_cols])
    record(f"Random forest ({tag})", tag, y_test, y_pred, y_proba, list(best.classes_))
    predictions[f"rf_{tag}"] = y_pred
    joblib.dump(best, MODELS / f"rf_{tag}.pkl")

    print(f"RF {tag:8s} best params: {search.best_params_}")
    print(f"RF {tag:8s} CV best acc: {search.best_score_:.3f}   Test acc: {accuracy_score(y_test, y_pred):.3f}")
    return best

rf_base = fit_rf(BASE_FEATURES, "base")
rf_enriched = fit_rf(ENRICHED_FEATURES, "enriched")

## 8 — Gradient boosting (`HistGradientBoostingClassifier`)

Gradient boosting þjálfar tré **hvert á eftir öðru í keðju**: sérhvert nýtt tré reynir að leiðrétta villur fyrri trjáa. `HistGradientBoostingClassifier` er nútímaleg útfærsla í Scikit-learn sem er hröð og ræður sjálfkrafa við `NaN`-gildi.

Við þjálfum þetta líkan einnig tvisvar (grunn og útvíkkað) með léttri leitargrind.

In [ ]:
def fit_histgb(feature_cols: list[str], tag: str):
    pipe = Pipeline([
        ("prep", build_preprocessor(feature_cols)),
        ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
    ])
    grid = {
        "clf__max_iter": [200, 500],
        "clf__learning_rate": [0.05, 0.1],
        "clf__max_leaf_nodes": [15, 31],
    }
    search = GridSearchCV(pipe, grid, cv=5, scoring="accuracy", n_jobs=-1)
    search.fit(train_df[feature_cols], y_train)
    best = search.best_estimator_

    y_pred = best.predict(test_df[feature_cols])
    y_proba = best.predict_proba(test_df[feature_cols])
    record(f"Gradient boosting ({tag})", tag, y_test, y_pred, y_proba, list(best.classes_))
    predictions[f"histgb_{tag}"] = y_pred
    joblib.dump(best, MODELS / f"histgb_{tag}.pkl")

    print(f"HistGB {tag:8s} best params: {search.best_params_}")
    print(f"HistGB {tag:8s} CV best acc: {search.best_score_:.3f}   Test acc: {accuracy_score(y_test, y_pred):.3f}")
    return best

histgb_base = fit_histgb(BASE_FEATURES, "base")
histgb_enriched = fit_histgb(ENRICHED_FEATURES, "enriched")

## 9 — AutoGluon (sjálfvirkt vélnám)

[AutoGluon](https://auto.gluon.ai/) er *AutoML*-safn sem þjálfar **mörg ólík líkön sjálfkrafa** (þar á meðal random forest, gradient boosting, k-nearest neighbors og tauganet), stillir þau og býr til **staflað safn** (*stacked ensemble*) úr þeim bestu.

Við gefum því 15 mínútur og segjum því einfaldlega að hámarka nákvæmni. Athugið að AutoGluon tekur gögnin **óhreinsuð** (með `NaN` og `Categorical`-gerðum) — það sér um alla forvinnslu sjálft.

Sama keyrsla tvisvar: grunn og útvíkkað.

**Ath:** þetta skref tekur ~30 mínútur alls (2×15). Niðurstöðurnar eru háðar því hversu öflug tölvan er sem þú notar til að keyra þetta forrit. Ef tölvan er öflugri færðu mögulega betri niðurstöður því hún getur framkvæmt fleiri útreikninga á 15 mínútum.

In [ ]:
from autogluon.tabular import TabularPredictor

AG_TIME = 15 * 60        # 15 minutes per run
AG_PRESETS = "medium_quality"

def fit_autogluon(feature_cols: list[str], tag: str):
    path = MODELS / f"autogluon_{tag}"
    # Start fresh each time; AutoGluon refuses to overwrite by default
    if path.exists():
        shutil.rmtree(path)

    train_subset = train_df[feature_cols + ["y"]].copy()
    test_subset = test_df[feature_cols + ["y"]].copy()

    predictor = TabularPredictor(
        label="y",
        eval_metric="accuracy",
        path=str(path),
        verbosity=1,
    ).fit(
        train_subset,
        time_limit=AG_TIME,
        presets=AG_PRESETS,
    )

    y_pred = predictor.predict(test_subset.drop(columns=["y"])).values
    y_proba_df = predictor.predict_proba(test_subset.drop(columns=["y"]))
    # Align prediction-probability columns to CLASS_ORDER
    classes_ag = list(y_proba_df.columns)
    y_proba = y_proba_df.values
    record(f"AutoGluon ({tag})", tag, y_test, y_pred, y_proba, classes_ag)
    predictions[f"autogluon_{tag}"] = y_pred

    print(f"\nAutoGluon {tag} leaderboard:")
    print(predictor.leaderboard().head(10))
    print(f"AutoGluon {tag} test accuracy: {accuracy_score(y_test, y_pred):.3f}")
    return predictor

ag_base = fit_autogluon(BASE_FEATURES, "base")
ag_enriched = fit_autogluon(ENRICHED_FEATURES, "enriched")

## 10 — Samantekt á niðurstöðum (fyrri umferð — *naive*)

Tökum saman niðurstöðurnar fyrir öll líkön í fyrri umferðinni. Seinna endurtökum við þjálfunina með leiðréttingu fyrir ójafnvægi flokka, og sömu
dálkar verða útfylltir fyrir þau líkön líka. Taflan er vistuð sem `figures/accuracy_table.csv` — og `figures/chapter_numbers.csv`.

In [ ]:
# Helper to render the results list as a human-friendly table with percentages
def results_to_display(res_list):
    df = pd.DataFrame(res_list)
    pct_cols = ["accuracy", "top2_accuracy", "top3_accuracy", "balanced_accuracy", "macro_f1"]
    for c in pct_cols:
        if c in df.columns:
            df[c] = (df[c] * 100).round(1)
    df = df.rename(columns={
        "model": "Líkan",
        "features": "Breytusafn",
        "treatment": "Meðhöndlun",
        "accuracy": "Nákvæmni (%)",
        "top2_accuracy": "Topp-2 (%)",
        "top3_accuracy": "Topp-3 (%)",
        "balanced_accuracy": "Jöfnuð nákv. (%)",
        "macro_f1": "Macro-F1 (%)",
    })
    return df

display_df = results_to_display(results)
# Persist snapshot for the naive pass; it is rewritten in section 20 to include
# the balanced pass too (both CSVs get the full table at the end).
display_df.to_csv(FIGURES / "accuracy_table.csv", index=False)
display_df.to_csv(FIGURES / "chapter_numbers.csv", index=False)
print("Samantekt á nákvæmni líkana (naive pass):")
display_df

## 11 — Ruglingsfylki (e. confusion matrix)

Ruglingsfylki sýnir **hvaða flokkum líkanið ruglar saman**. Raðir tákna rétt flokksval, dálkar tákna forspá líkansins. Sérhver reitur sýnir hlutfall flokkaðra svara. Athugið að fyrstu tveir dálkarnir sýna að líkönin hafa sterka tilhneigingu til að giska á Samfylkinguna eða Sjálfstæðisflokkinn.

In [ ]:
def plot_cm(y_true, y_pred, title: str, out_path: Path):
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER, normalize="true")
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=CLASS_LABELS_IS, yticklabels=CLASS_LABELS_IS,
        cbar_kws={"label": "Hlutfall (raðnormalíserað)"},
        ax=ax,
    )
    ax.set_xlabel("Forspá líkans")
    ax.set_ylabel("Raunverulegt flokksval")
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()

# Plot confusion matrices for the three "interesting" models on base features
plot_cm(y_test, predictions["rf_base"],           "Random forest (grunn)",      FIGURES / "confusion_matrix_rf_base.png")
plot_cm(y_test, predictions["rf_enriched"],       "Random forest (útvíkkað)",   FIGURES / "confusion_matrix_rf_enriched.png")
plot_cm(y_test, predictions["histgb_base"],       "Gradient boosting (grunn)",  FIGURES / "confusion_matrix_histgb_base.png")
plot_cm(y_test, predictions["autogluon_base"],    "AutoGluon (grunn)",          FIGURES / "confusion_matrix_autogluon_base.png")
plot_cm(y_test, predictions["autogluon_enriched"],"AutoGluon (útvíkkað)",       FIGURES / "confusion_matrix_autogluon_enriched.png")

## 12 — Flokkasérhæfð nákvæmni (*per-class precision/recall*)

Random forest á grunnbreytum er fyrirmyndarlíkan fyrir SHAP-túlkun í þriðja skjalinu. Hér er flokkasérhæfð samantekt fyrir það.

In [ ]:
rep = classification_report(
    y_test, predictions["rf_base"], labels=CLASS_ORDER,
    target_names=CLASS_LABELS_IS, output_dict=True, zero_division=0,
)
rep_df = pd.DataFrame(rep).T.round(3)
rep_df.to_csv(FIGURES / "classification_report_rf_base.csv")
rep_df

## 13 — Vistum forspár líkananna

Við vistum forspár allra líkana (og raunveruleg gildi) í eina parquet-skrá svo þriðja skjalið geti hlaðið þeim á ný án þess að endurreikna allt.

In [ ]:
pred_df = pd.DataFrame({f"pred_{k}": v for k, v in predictions.items()})
pred_df.insert(0, "y_true", y_test.values)
pred_df.index = y_test.index
pred_df.to_parquet(PROC / "predictions.parquet")
print(f"Wrote {PROC / 'predictions.parquet'} ({pred_df.shape})")
pred_df.head()

## 14 — Hvers vegna er fyrri umferð skökk?

Þegar við skoðum **flokkasérhæfða endurköllun** (*per-class recall*) fyrir random forest á grunnbreytum í fyrri umferðinni sjáum við mjög ójafna
frammistöðu:

| Flokkur | Fjöldi í prófunarúrtaki | Recall |
|---|---|---|
| Samfylkingin (stærst) | 115 | ~70% |
| Sjálfstæðisflokkur | 90 | ~71% |
| Flokkur fólksins | 49 | **~18%** |
| Vinstri (sameinað) | 50 | ~26% |

Líkanið sýnir **hlutdrægni í átt að stærstu flokkunum**. Þetta er klassískt vandamál þegar flokkar eru misstórir í gögnunum: líkanið hámarkar heildar-
nákvæmni með því að giska á stóru flokkana oftar. Fyrir vikið giskar það síður á litlu flokkana.

Hér fyrir neðan endurtökum við þjálfunina með **leiðréttingu fyrir ójafnvægi**. Við gerum það með því að **breyta vægi flokka** (*class weighting*) í þjálfuninni. Þannig segjum við líkaninu að mistök á litlum flokkum séu jafn kostnaðarsöm og mistök á stórum flokkum.

**Tvær aðalaðferðir:**

- **`class_weight="balanced"`** — fyrir `DecisionTreeClassifier` og `RandomForestClassifier`. Ein stillingarbreyta: vægið er í öfugu hlutfalli við flokkstíðni.
- **`sample_weight`** — fyrir `HistGradientBoostingClassifier` og AutoGluon, sem hafa ekki beinan `class_weight`-stika. Við notum samt sem áður sömu nálgun og áður því við reiknum út vigt fyrir hvert og eitt dæmi sem er í öfugu hlutfalli við algengi flokksins sem viðkomandi aðili kaus.

**Mælikvarði sem við notum til að meta árangurinn:** *jöfnuð nákvæmni* (e. *balanced accuracy*) — meðaltal recall yfir flokkana sjö. Þetta er aðalmælikvarði fyrir ójafnvæg gögn og sýnir beint hvort líkanið finnur litlu flokkana.

In [ ]:
# Show the baseline pass's per-class recall for rf_base — the motivation number
_naive_rf_recall = [
    r for r in per_class_recall_rows
    if r["model"] == "Random forest (base)" and r["treatment"] == "naive"
]
_naive_df = pd.DataFrame(_naive_rf_recall).assign(
    recall_pct=lambda d: (d["recall"] * 100).round(1)
)[["class", "recall_pct"]]
_naive_df["party"] = _naive_df["class"].map(tl["target_class_labels_is"])
print("Naive-pass per-class recall fyrir rf_base (motivation):")
print(_naive_df[["party", "recall_pct"]].to_string(index=False))

## 15 — Ákvarðanatré með `class_weight="balanced"`

Fyrst endurþjálfum við ákvarðanatréð með `class_weight="balanced"`. Þetta er ein lína breytt í skilgreiningunni, líkanið reiknar sjálft út að minni flokkar skuli hafa hærra vægi í þjálfunargögnunum.

In [ ]:
tree_pipe_bal = Pipeline([
    ("prep", build_preprocessor(BASE_FEATURES)),
    ("clf", DecisionTreeClassifier(
        max_depth=3,
        class_weight="balanced",   # <-- the only real change
        random_state=RANDOM_STATE,
    )),
])
tree_pipe_bal.fit(train_df[BASE_FEATURES], y_train)
tree_pred_bal = tree_pipe_bal.predict(test_df[BASE_FEATURES])
tree_proba_bal = tree_pipe_bal.predict_proba(test_df[BASE_FEATURES])

record(
    "Decision tree (max_depth=3)", "base",
    y_test, tree_pred_bal, tree_proba_bal, list(tree_pipe_bal.classes_),
    treatment="balanced",
)
predictions["decision_tree_balanced"] = tree_pred_bal
joblib.dump(tree_pipe_bal, MODELS / "decision_tree_balanced.pkl")
print(f"Decision tree (balanced) accuracy          : {accuracy_score(y_test, tree_pred_bal):.3f}")
print(f"Decision tree (balanced) balanced accuracy : {balanced_accuracy_score(y_test, tree_pred_bal):.3f}")

## 16 — Random forest með leiðréttu ójafnvægi

Sama aðferð fyrir random forest. Við þjálfum aftur á báðum breytusöfnum (grunn + útvíkkað) með `class_weight="balanced"`. Notuð er sama leitargrind og í fyrri umferð — en **val á bestu stikum er nú gert með jafnaðri nákvæmni** (`scoring="balanced_accuracy"`), í samræmi við markmið þessarar umferðar.

In [ ]:
def fit_rf_balanced(feature_cols: list[str], tag: str):
    """Random forest with class_weight='balanced', tuned on balanced accuracy."""
    pipe = Pipeline([
        ("prep", build_preprocessor(feature_cols)),
        ("clf", RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])
    grid = {
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_leaf": [1, 5, 10],
    }
    search = GridSearchCV(pipe, grid, cv=5, scoring="balanced_accuracy", n_jobs=-1)
    search.fit(train_df[feature_cols], y_train)
    best = search.best_estimator_

    y_pred = best.predict(test_df[feature_cols])
    y_proba = best.predict_proba(test_df[feature_cols])
    record(
        f"Random forest ({tag})", tag,
        y_test, y_pred, y_proba, list(best.classes_),
        treatment="balanced",
    )
    predictions[f"rf_{tag}_balanced"] = y_pred
    joblib.dump(best, MODELS / f"rf_{tag}_balanced.pkl")

    print(f"RF {tag:8s} balanced — best params: {search.best_params_}")
    print(f"RF {tag:8s} balanced — CV bal. acc: {search.best_score_:.3f}   "
          f"Test bal. acc: {balanced_accuracy_score(y_test, y_pred):.3f}")
    return best

rf_base_balanced = fit_rf_balanced(BASE_FEATURES, "base")
rf_enriched_balanced = fit_rf_balanced(ENRICHED_FEATURES, "enriched")

## 17 — Gradient boosting með `sample_weight`

`HistGradientBoostingClassifier` tekur ekki við `class_weight`-stikanum. Þess í stað breytum við **vægi hvers dæmis** (`sample_weight`) sem jafngildir `class_weight="balanced"` þegar vigtirnar eru reiknaðar með `compute_sample_weight("balanced", y_train)`. Þetta sýnir dæmigerð mynstur í Scikit-learn pakkanum, þ.e., ekki öll líkön hafa sama viðmót til að stilla þau, en hugmyndin er sú sama.

Við látum vigtirnar fljóta í gegn um `GridSearchCV` með því að gefa þær með forskeytinu `clf__sample_weight`.

In [ ]:
def fit_histgb_balanced(feature_cols: list[str], tag: str):
    """HistGradientBoosting with sample_weight, tuned on balanced accuracy."""
    pipe = Pipeline([
        ("prep", build_preprocessor(feature_cols)),
        ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
    ])
    grid = {
        "clf__max_iter": [200, 500],
        "clf__learning_rate": [0.05, 0.1],
        "clf__max_leaf_nodes": [15, 31],
    }
    search = GridSearchCV(pipe, grid, cv=5, scoring="balanced_accuracy", n_jobs=-1)
    # sample_weight flows through GridSearchCV to the classifier inside each CV fold
    search.fit(
        train_df[feature_cols], y_train,
        clf__sample_weight=sample_weight_balanced,
    )
    best = search.best_estimator_

    y_pred = best.predict(test_df[feature_cols])
    y_proba = best.predict_proba(test_df[feature_cols])
    record(
        f"Gradient boosting ({tag})", tag,
        y_test, y_pred, y_proba, list(best.classes_),
        treatment="balanced",
    )
    predictions[f"histgb_{tag}_balanced"] = y_pred
    joblib.dump(best, MODELS / f"histgb_{tag}_balanced.pkl")

    print(f"HistGB {tag:8s} balanced — best params: {search.best_params_}")
    print(f"HistGB {tag:8s} balanced — CV bal. acc: {search.best_score_:.3f}   "
          f"Test bal. acc: {balanced_accuracy_score(y_test, y_pred):.3f}")
    return best

histgb_base_balanced = fit_histgb_balanced(BASE_FEATURES, "base")
histgb_enriched_balanced = fit_histgb_balanced(ENRICHED_FEATURES, "enriched")

## 18 — AutoGluon með leiðréttu ójafnvægi

AutoGluon tekur við dálki með vigtum í sjálfum þjálfunarramma gagnanna. Við bætum honum við, tilgreinum `sample_weight="sample_weight"` í `TabularPredictor`, og förum fram á `eval_metric="balanced_accuracy"` til að AutoGluon velji besta líkanið út frá jafnaðri nákvæmni.

`weight_evaluation=False` tryggir að **prófunarnákvæmnin okkar er án vigta**, þannig er samanburður við naive-umferðina sanngjarn.

**Athugið:** þetta skref tekur aftur ~30 mínútur alls (2×15).

In [ ]:
def fit_autogluon_balanced(feature_cols: list[str], tag: str):
    path = MODELS / f"autogluon_{tag}_balanced"
    if path.exists():
        shutil.rmtree(path)

    # Add the sample_weight column AutoGluon needs
    train_subset = train_df[feature_cols + ["y"]].copy()
    train_subset["sample_weight"] = sample_weight_balanced
    test_subset = test_df[feature_cols + ["y"]].copy()

    predictor = TabularPredictor(
        label="y",
        sample_weight="sample_weight",
        weight_evaluation=False,              # test metrics computed unweighted
        eval_metric="balanced_accuracy",
        path=str(path),
        verbosity=1,
    ).fit(
        train_subset,
        time_limit=AG_TIME,
        presets=AG_PRESETS,
    )

    y_pred = predictor.predict(test_subset.drop(columns=["y"])).values
    y_proba_df = predictor.predict_proba(test_subset.drop(columns=["y"]))
    classes_ag = list(y_proba_df.columns)
    y_proba = y_proba_df.values
    record(
        f"AutoGluon ({tag})", tag,
        y_test, y_pred, y_proba, classes_ag,
        treatment="balanced",
    )
    predictions[f"autogluon_{tag}_balanced"] = y_pred

    print(f"\nAutoGluon {tag} balanced — leaderboard head:")
    print(predictor.leaderboard().head(10))
    print(f"AutoGluon {tag} balanced — test bal. acc: "
          f"{balanced_accuracy_score(y_test, y_pred):.3f}")
    return predictor

ag_base_balanced = fit_autogluon_balanced(BASE_FEATURES, "base")
ag_enriched_balanced = fit_autogluon_balanced(ENRICHED_FEATURES, "enriched")

## 19 — Samanburður: *naive* vs. *balanced*

Tvær sýnilegar breytingar:

1. **Heildarnákvæmni** (*accuracy*) getur lækkað lítillega vegna þess að líkanið fórnar einhverjum réttum svörum í stóru flokkunum til að hitta
   á litlu flokkana.
2. **Jöfnuð nákvæmni** og **macro-F1** hækka oft umtalsvert en þau eru þeir mælikvarðar sem við viljum hámarka þegar gögn eru í ójafnvægi.

Fyrir neðan: samanburðartafla yfir allar umferðir og mynd sem sýnir **endurheimt fyrir hvern flokk** í báðum umferðum á `rf_base` (sem er líkanið sem við túlkum með SHAP í næstu minnisbók).

In [ ]:
# Full comparison table (naive + balanced, every model)
display_df_full = results_to_display(results)
print("Samanburðartafla (báðar umferðir):")
display_df_full

In [ ]:
# Per-class recall — naive vs balanced for rf_base, as a grouped bar chart
recall_df = pd.DataFrame(per_class_recall_rows)

# Focus on RF base (the SHAP target)
rf_base_recall = recall_df[
    (recall_df["model"] == "Random forest (base)") &
    (recall_df["features"] == "base")
].copy()
rf_base_recall["party"] = rf_base_recall["class"].map(tl["target_class_labels_is"])

# Sort by naive recall so bars are easy to read
order = (
    rf_base_recall[rf_base_recall["treatment"] == "naive"]
    .sort_values("recall", ascending=False)["party"].tolist()
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=rf_base_recall, x="party", y="recall", hue="treatment",
    order=order, hue_order=["naive", "balanced"],
    palette={"naive": "#c75b5b", "balanced": "#4a90a4"},
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("Recall (per-class)")
ax.set_title("Random forest (grunn): flokkasérhæfð recall — naive vs. balanced")
ax.set_ylim(0, 1)
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{h:.2f}", (p.get_x() + p.get_width() / 2, h),
                    ha="center", va="bottom", fontsize=8)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
fig.savefig(FIGURES / "per_class_recall_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 20 — Vista lokaútkomur (báðar umferðir)

Loks vistum við allt sem þriðja skjalið þarf til að bera saman umferðirnar:

- Uppfærð `accuracy_table.csv` og `chapter_numbers.csv` með báðum umferðum.
- `per_class_recall.csv` — löng tafla (`model × features × treatment × class → recall`).
- Ruglingsyfirlit fyrir þýðingarmestu jöfnuðu líkönin.
- Flokkunarskýrsla (`classification_report`) fyrir `rf_base_balanced`.
- Uppfærð `predictions.parquet` með spám úr báðum umferðum.

In [ ]:
# Final rewrite of the accuracy table CSVs (now includes both passes)
display_df_full = results_to_display(results)
display_df_full.to_csv(FIGURES / "accuracy_table.csv", index=False)
display_df_full.to_csv(FIGURES / "chapter_numbers.csv", index=False)

# Long-format per-class recall CSV
recall_out = pd.DataFrame(per_class_recall_rows).copy()
recall_out["party"] = recall_out["class"].map(tl["target_class_labels_is"])
recall_out["recall_pct"] = (recall_out["recall"] * 100).round(1)
recall_out.to_csv(FIGURES / "per_class_recall.csv", index=False)
print(f"Wrote {FIGURES / 'per_class_recall.csv'} ({recall_out.shape})")

In [ ]:
# Confusion matrices for the balanced pass's key models
plot_cm(y_test, predictions["rf_base_balanced"],
        "Random forest (grunn, balanced)",
        FIGURES / "confusion_matrix_rf_base_balanced.png")
plot_cm(y_test, predictions["rf_enriched_balanced"],
        "Random forest (útvíkkað, balanced)",
        FIGURES / "confusion_matrix_rf_enriched_balanced.png")
plot_cm(y_test, predictions["histgb_base_balanced"],
        "Gradient boosting (grunn, balanced)",
        FIGURES / "confusion_matrix_histgb_base_balanced.png")
plot_cm(y_test, predictions["autogluon_base_balanced"],
        "AutoGluon (grunn, balanced)",
        FIGURES / "confusion_matrix_autogluon_base_balanced.png")
plot_cm(y_test, predictions["autogluon_enriched_balanced"],
        "AutoGluon (útvíkkað, balanced)",
        FIGURES / "confusion_matrix_autogluon_enriched_balanced.png")

In [ ]:
# Classification report for the balanced rf_base — pairs with the naive one saved in section 12
rep_bal = classification_report(
    y_test, predictions["rf_base_balanced"], labels=CLASS_ORDER,
    target_names=CLASS_LABELS_IS, output_dict=True, zero_division=0,
)
rep_bal_df = pd.DataFrame(rep_bal).T.round(3)
rep_bal_df.to_csv(FIGURES / "classification_report_rf_base_balanced.csv")
rep_bal_df

In [ ]:
# Final predictions.parquet — supersedes the write in section 13, now with balanced cols
pred_df = pd.DataFrame({f"pred_{k}": v for k, v in predictions.items()})
pred_df.insert(0, "y_true", y_test.values)
pred_df.index = y_test.index
pred_df.to_parquet(PROC / "predictions.parquet")
print(f"Wrote {PROC / 'predictions.parquet'} ({pred_df.shape})")
print("Columns:", list(pred_df.columns))

## Samantekt

**Fyrri umferð (*naive*)** og **seinni umferð (*balanced*)** eru nú báðar
vistaðar. Seinni umferðin hámarkar **jöfnaða nákvæmni** með því að þyngja
litla flokka í þjálfuninni.

| Vistaðar skrár | Innihald |
|---|---|
| `models/preprocessor_{base,enriched}.pkl` | Forvinnsluleiðslur |
| `models/decision_tree{,_balanced}.pkl` | Ákvarðanatré, báðar umferðir |
| `models/rf_{base,enriched}{,_balanced}.pkl` | Random forest, báðar umferðir |
| `models/histgb_{base,enriched}{,_balanced}.pkl` | Gradient boosting, báðar umferðir |
| `models/autogluon_{base,enriched}{,_balanced}/` | AutoGluon, báðar umferðir |
| `data/processed/predictions.parquet` | Forspár allra líkana, báðar umferðir |
| `figures/decision_tree.png`, `confusion_matrix_*.png` | Myndir |
| `figures/per_class_recall_comparison.png` | Grafísk samanburðarmynd (naive vs balanced) |
| `figures/per_class_recall.csv` | Löng tafla per-class recall |
| `figures/accuracy_table.csv`, `chapter_numbers.csv` | Allt sem kaflatextinn þarf |

**Næst:** `03_shap.ipynb` ber saman SHAP-túlkun á `rf_base` (naive) og
`rf_base_balanced` — sýnir hvort og hvernig leiðréttingin breytir því
**hvaða breytur** líkanið styðst við.